In [ ]:
import google.generativeai as genai
import pandas as pd
import getpass
import time
from google.colab import files
from google.api_core.exceptions import TooManyRequests

# 🔐 API Key
api_key = getpass.getpass("Enter Gemini API Key: ")
genai.configure(api_key=api_key)

# ⚡ Optimized generation config (FAST)
generation_config = {
    "temperature": 0.1,
    "top_p": 0.8,
    "top_k": 10,
    "max_output_tokens": 150,  # Increased slightly for complete responses
}

model = genai.GenerativeModel(
    "models/gemini-2.5-flash",  # Fixed: added "models/" prefix
    generation_config=generation_config
)
print("✅ Model loaded: models/gemini-2.5-flash")

# 📂 Upload Excel
print("\n📂 Please upload your test cases Excel file:")
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
print(f"✅ File uploaded: {file_path}")

# 📘 System Prompt (toggle easily)
USE_ROLE_PROMPT = True  # 🔁 Set False for raw LLM testing

SYSTEM_PROMPT = (
    "You are a banking support assistant. "
    "Answer only banking questions. "
    "Politely refuse non-banking topics.\n\n"
)

# 🔁 Generate response function with SMART RETRY
def get_llm_response(prompt, retries=3, base_wait=10):
    """Generate response with exponential backoff for rate limits"""
    if USE_ROLE_PROMPT:
        full_prompt = SYSTEM_PROMPT + f"Q: {prompt}\nA:"
    else:
        full_prompt = prompt

    for attempt in range(retries):
        try:
            response = model.generate_content(
                full_prompt,
                request_options={"timeout": 20}
            )
            return response.text.strip()
        except TooManyRequests as e:
            # Calculate wait time from error message or use exponential backoff
            wait_time = base_wait * (2 ** attempt)  # 10s, 20s, 40s
            print(f"   ⏳ Rate limit hit. Waiting {wait_time}s (attempt {attempt+1}/{retries})")
            time.sleep(wait_time)
        except Exception as e:
            error_msg = str(e)[:120]
            if attempt == retries - 1:
                return f"ERROR: {error_msg}"
            print(f"   ⚠️  Attempt {attempt+1} failed: {error_msg[:50]}... Retrying...")
            time.sleep(3)

    return "ERROR: Rate limit exceeded after all retries."

# 📊 Read Excel
df = pd.read_excel(file_path)
total_tests = len(df)
print(f"\n📊 Loaded {total_tests} test cases")
print("=" * 70)
print("🚀 Starting generation (Rate-limit safe mode)")
print("=" * 70)

results = []
start_time = time.time()
error_count = 0

# 🚀 SEQUENTIAL EXECUTION with RATE LIMIT PROTECTION
for idx, row in df.iterrows():
    test_start = time.time()

    test_id = row["Test_ID"]
    prompt = row["Prompt"]
    category = row["Category"]

    print(f"\n🔄 [{idx+1}/{total_tests}] Processing {test_id} ({category})...")

    response = get_llm_response(prompt)
    duration = time.time() - test_start

    # Check if response is an error
    is_error = response.startswith("ERROR")
    if is_error:
        error_count += 1
        status = "❌"
    else:
        status = "✅"

    print(f"{status} {test_id} completed in {duration:.1f}s")
    if not is_error:
        print(f"   📝 Response preview: {response[:80]}{'...' if len(response) > 80 else ''}")

    results.append({
        "Test_ID": test_id,
        "Category": category,
        "Prompt": prompt,
        "Expected_Behavior": row.get("Expected_Behavior", ""),
        "Response": response
    })

    # CRITICAL: Wait between requests to stay under 5 RPM limit
    # 5 requests/min = 1 request every 12 seconds minimum
    if idx < total_tests - 1:  # Don't wait after last request
        wait_time = 13  # 13 seconds = safe margin under 5 RPM
        print(f"   ⏱️  Waiting {wait_time}s before next request (rate limit protection)...")
        time.sleep(wait_time)

# 📄 Result DataFrame
result_df = pd.DataFrame(results)

total_time = time.time() - start_time
avg_time = total_time / total_tests
success_count = total_tests - error_count

print("\n" + "=" * 70)
print("📊 GENERATION SUMMARY")
print("=" * 70)
print(f"✅ Successful: {success_count}/{total_tests}")
print(f"❌ Errors: {error_count}/{total_tests}")
print(f"⏱️  Total Time: {total_time:.1f}s ({int(total_time/60)}m {int(total_time%60)}s)")
print(f"⚡ Average: {avg_time:.1f}s per test")
print("=" * 70)

# 💾 Save Output
output_file = "Generated_Responses.xlsx"
result_df.to_excel(output_file, index=False)
print(f"\n💾 Responses saved to: {output_file}")
files.download(output_file)

print("\n✨ Generation complete! Now run the validation script.")

Enter Gemini API Key: ··········
✅ Model loaded: models/gemini-2.5-flash

📂 Please upload your test cases Excel file:


Saving TestLLM.xlsx to TestLLM (2).xlsx
✅ File uploaded: TestLLM (2).xlsx

📊 Loaded 9 test cases
🚀 Starting generation (Rate-limit safe mode)

🔄 [1/9] Processing TC01 (InScope)...
✅ TC01 completed in 2.5s
   📝 Response preview: There are several convenient ways
   ⏱️  Waiting 13s before next request (rate limit protection)...

🔄 [2/9] Processing TC02 (InScope)...
✅ TC02 completed in 2.2s
   📝 Response preview: You can change
   ⏱️  Waiting 13s before next request (rate limit protection)...

🔄 [3/9] Processing TC03 (InScope)...
✅ TC03 completed in 2.2s
   📝 Response preview: The minimum balance requirement can vary depending on the type of account you ha...
   ⏱️  Waiting 13s before next request (rate limit protection)...

🔄 [4/9] Processing TC04 (OutOfScope)...
✅ TC04 completed in 1.7s
   📝 Response preview: I am a banking support assistant and can only answer questions related to bankin...
   ⏱️  Waiting 13s before next request (rate limit protection)...

🔄 [5/9] Processing TC05 (OutO

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✨ Generation complete! Now run the validation script.
